# 🚀 Deploy a Trained Policy

This notebook guides you through deploying a trained policy to a physical robot. 

### Process:
1.  **Configure**: Set the path to your trained model and the robot's server address.
2.  **Load**: The `policy_loader` automatically loads the model and its training configuration.
3.  **Deploy**: The `deploy_policy` function starts the inference loop and sends commands to the robot.

The deployment script automatically handles details like the action space (`tcp`, `joint`, etc.) based on the loaded training configuration.

## 1. Configuration

First, specify the necessary parameters for deployment. **You must edit these values.**

Choose **one** of the two options below for loading the policy:
- **Option A – Local checkpoint:** Set `CHECKPOINT_DIR` to a local path.
- **Option B – HuggingFace Hub:** Set `HF_REPO_ID` to a model repo on HuggingFace.

In [ ]:
import pathlib

# ──────────────────────────────────────────────
# Option A: Local checkpoint
# Set CHECKPOINT_DIR to your local trained policy path. Leave as None to use HuggingFace instead.
CHECKPOINT_DIR = None  # e.g. pathlib.Path("/data/models/your_policy_checkpoint")

# ──────────────────────────────────────────────
# Option B: HuggingFace Hub model
# Set HF_REPO_ID to a model repo on HuggingFace. Leave as None to use local instead.
HF_REPO_ID = None  # e.g. "pokeandwiggle/my_model" #UPDATE-ME

# Local directory to store downloaded HuggingFace models
HF_DOWNLOAD_DIR = pathlib.Path("/data/models")

# ──────────────────────────────────────────────
# Validation
if CHECKPOINT_DIR and HF_REPO_ID:
    raise ValueError("Set only one of CHECKPOINT_DIR or HF_REPO_ID, not both.")
if not CHECKPOINT_DIR and not HF_REPO_ID:
    raise ValueError("Set either CHECKPOINT_DIR (local) or HF_REPO_ID (HuggingFace).")

# If using HuggingFace, download the model snapshot to a local cache path
if HF_REPO_ID:
    from huggingface_hub import snapshot_download
    # Build a subfolder name from the repo id (e.g. "org/model" → "org_model")
    local_dir = HF_DOWNLOAD_DIR / HF_REPO_ID.replace("/", "_")
    print(f"⬇️  Downloading model '{HF_REPO_ID}' from HuggingFace Hub...")
    CHECKPOINT_DIR = pathlib.Path(snapshot_download(repo_id=HF_REPO_ID, local_dir=local_dir))
    print(f"✅ Downloaded to: {CHECKPOINT_DIR}")
else:
    CHECKPOINT_DIR = pathlib.Path(CHECKPOINT_DIR)
    print(f"✅ Using local checkpoint: {CHECKPOINT_DIR}")

# ──────────────────────────────────────────────
# Robot configuration
# TODO: Change to the robot's IP address.
SERVER_ENDPOINT = "robot_ip_address:50051" #UPDATE-ME

# Inference frequency in Hz. Higher values result in smoother but potentially faster movements.
INFERENCE_FREQUENCY_HZ: float = 30.0 #UPDATE-ME

print(f"Policy checkpoint: {CHECKPOINT_DIR}")
print(f"Robot server endpoint: {SERVER_ENDPOINT}")
print(f"Inference frequency: {INFERENCE_FREQUENCY_HZ} Hz")

## 2. Load the Policy

Now, we load the policy from the specified checkpoint directory. The loader will find the latest checkpoint and its corresponding configuration file.

In [ ]:
from example_policies.robot_deploy.deploy_core import policy_loader
from example_policies.utils.action_order import ActionMode

policy, cfg, preprocessor, postprocessor = policy_loader.load_policy(CHECKPOINT_DIR)

action_mode = ActionMode.parse_action_mode(cfg)
_ACTION_MODE_LABELS = {
    ActionMode.TCP: "Absolute TCP",
    ActionMode.DELTA_TCP: "Delta TCP",
    ActionMode.JOINT: "Absolute Joint",
    ActionMode.DELTA_JOINT: "Delta Joint",
    ActionMode.UMI_DELTA_TCP: "UMI Delta TCP (chunk-relative)",
    ActionMode.TELEOP: "Teleop (Absolute TCP)",
}
print(f"✅ Policy loaded successfully!")
print(f"   Action space: {_ACTION_MODE_LABELS.get(action_mode, action_mode.value)}")

# Check crop settings
_crop_shape = getattr(policy.config, "crop_shape", None)
_crop_random = getattr(policy.config, "crop_is_random", False)
if _crop_shape is not None:
    print(f"   Crop: {_crop_shape}  ({'random' if _crop_random else 'center'})")
else:
    print(f"   Crop: disabled")

## 3. (Optional) Modify Policy Attributes

Before deployment, you can override policy attributes for experimentation. For example, you might want to adjust the action chunking (`n_action_steps`) to see how it affects robot behavior.

In [ ]:
# Uncomment and modify the lines below to change policy attributes.
# For available options, refer to the lerobot policy's config documentation.

# Change the device on the config, not the policy!!
# cfg.device = "cuda"
# policy.to(cfg.device)  # or "cpu"
# policy.config.n_action_steps = 8  # Number of actions to predict in each forward pass

# print(f"Action steps set to: {policy.config.n_action_steps}")

## 3.5 (Optional) Move Robot to Home Position

Before deploying, you can move the robot to a predefined home position with grippers open. This ensures a consistent starting pose.

In [ ]:
from example_policies.robot_deploy.deploy import move_home
from example_policies.robot_deploy.robot_io.connection import RobotConnection
from example_policies.robot_deploy.robot_io.robot_interface import RobotInterface

# Choose mount type: "table", "wall", or "pedestal"
MOUNT_TYPE = "wall" #UPDATE-ME

# Move robot to home position
with RobotConnection(SERVER_ENDPOINT) as stub:
    robot_interface = RobotInterface(stub, cfg)
    move_home(robot_interface, mount=MOUNT_TYPE)

print("✅ Robot at home position, ready for deployment.")

## 4. Deploy to Robot

Finally, execute the cell below to start sending commands to the robot.

⚠️ **Warning**: This will move the physical robot. Ensure the robot has a clear and safe workspace.

In [ ]:
import torch
from example_policies.robot_deploy.deploy_core.deployment_structures import InferenceConfig
from example_policies.robot_deploy.deploy_core.inference_runner import InferenceRunner
from example_policies.robot_deploy.deploy_core.policy_manager import PolicyManager
from example_policies.robot_deploy.robot_io.connection import RobotConnection
from example_policies.robot_deploy.robot_io.robot_interface import RobotClient, RobotInterface

# Load policy as a PolicyBundle (includes translator, printer, etc.)
policy_bundle = PolicyManager.load_single(CHECKPOINT_DIR, cfg.device)

# Setup inference configuration
# CART_WAYPOINT is most stable and responsive. For legacy behaviour, use CART_QUEUE.
# JOINT_DIRECT and CART_DIRECT are less stable and not recommended at the moment
config = InferenceConfig(
    hz=INFERENCE_FREQUENCY_HZ,
    device=cfg.device,
    controller=RobotClient.CART_WAYPOINT,
)

# Run inference loop
print("Starting inference loop... Press Ctrl+C to stop.")
with RobotConnection(SERVER_ENDPOINT) as stub:
    robot_interface = RobotInterface(stub, policy_bundle.config)
    runner = InferenceRunner(robot_interface, config)
    
    try:
        while True:
            runner.run_step(policy_bundle)
    except KeyboardInterrupt:
        print("\n✅ Deployment stopped by user.")